# Diabetes Prediction Project

A structured machine learning pipeline for diabetes classification.

**Steps covered:**
1. Import Packages
2. Load Dataset
3. Data Cleaning
4. Exploratory Data Analysis
5. Feature Engineering
6. Model Training and Evaluation
7. Save Artifacts

---
## 1. Import Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

import joblib

---
## 2. Load Dataset

In [ ]:
df = pd.read_csv(r"U:\Universiy\Level_2-S_2\AI\Project\diabetes_dataset last data.csv")
df_copy = df.copy()
df_copy.head()

In [ ]:
df_copy.info()

In [ ]:
df_copy.describe()

---
## 3. Data Cleaning

### 3.1 Missing Values

In [ ]:
df_copy.isna().mean() * 100

### 3.2 Remove Duplicates

In [ ]:
print(df_copy.duplicated().sum())
df_copy = df_copy.drop_duplicates()

In [ ]:
# Confirm duplicates removed
print(df_copy.duplicated().sum())

---
## 4. Exploratory Data Analysis (EDA)

### 4.1 Target Distribution

In [ ]:
sns.countplot(x='diabetes', data=df_copy)
plt.show()

### 4.2 Gender Distribution

In [ ]:
sns.histplot(x='gender', data=df_copy)

### 4.3 Numerical Features - Boxplots

In [ ]:
cols = list(df_copy.select_dtypes(['int64', 'float64']))
for col in cols:
    sns.boxplot(y=col, data=df_copy)
    plt.show()

---
## 5. Feature Engineering

### 5.1 Drop Irrelevant Columns

In [ ]:
df_copy = df_copy.drop(
    columns=['year', 'location', 'race:AfricanAmerican', 'race:Asian',
             'race:Caucasian', 'race:Hispanic', 'race:Other']
)

In [ ]:
df_copy.info()

In [ ]:
df_copy.describe()

In [ ]:
df_copy.head()

### 5.2 Correlation Heatmap

In [ ]:
num_data = df_copy.select_dtypes(['int64', 'float64'])
corr1 = num_data.corr()
sns.heatmap(corr1, annot=True, cmap="YlGnBu")
plt.show()

### 5.3 Encode Categorical Variables

**Gender** - binary, use Label Encoding

In [ ]:
le = LabelEncoder()
df_copy['gender'] = le.fit_transform(df_copy['gender'])
df_copy['gender']

**Smoking History** - multi-class, use One-Hot Encoding

In [ ]:
df_copy = df_copy.reset_index(drop=True)

OHE = OneHotEncoder()
ohe_encoder = OHE.fit_transform(df_copy[['smoking_history']])
ohe_encoder_df = pd.DataFrame(
    ohe_encoder.toarray(),
    columns=OHE.get_feature_names_out(['smoking_history'])
)

df_copy = pd.concat([df_copy.drop('smoking_history', axis=1), ohe_encoder_df], axis=1)
df_copy.head()

In [ ]:
# Confirm no missing values after encoding
df_copy.isna().mean() * 100

---
## 6. Train-Test Split

In [ ]:
X = df_copy.drop('diabetes', axis=1)
y = df_copy['diabetes']

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=42)

---
## 7. Model Training

### 7.1 Logistic Regression (Baseline)

In [ ]:
logi = LogisticRegression()
logi.fit(X_train, y_train)

In [ ]:
y_pred_train = logi.predict(X_train)

In [ ]:
y_pred_logi = logi.predict(X_test)

print("Logistic Regression Performance:")
print(classification_report(y_test, y_pred_logi))

plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_logi), annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix: Logistic Regression')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

### 7.2 Random Forest and XGBoost

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# XGBoost
xgb = XGBClassifier()
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

print("Models trained successfully.")

---
## 8. Model Comparison

In [ ]:
models_comparison = {
    "Logistic Regression": y_pred_logi,
    "Random Forest":       y_pred_rf,
    "XGBoost":             y_pred_xgb
}

for name, preds in models_comparison.items():
    print(f"\n--- {name} ---")
    print(f"F1-Score : {f1_score(y_test, preds):.4f}")
    print(f"Recall   : {recall_score(y_test, preds):.4f}")

---
## 9. Save Artifacts

In [ ]:
joblib.dump(xgb, 'diabetes_model_final.pkl')
joblib.dump(le,  'gender_encoder.pkl')
joblib.dump(OHE, 'smoking_ohe.pkl')

print("All artifacts saved.")

---
## 10. Feature Documentation

In [ ]:
print("Final feature list (input order for prediction):")
for i, column in enumerate(X.columns):
    print(f"  {i}: {column}")